In [1]:
import os
import shutil
import random
import numpy as np
import librosa
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow import keras
from tensorflow.keras import layers, callbacks

In [2]:
source_dir = "../data/Voice_Dataset"
target_dir = "../data/parkinson"

class_map = {
    "Healthy": "Healthy",
    "Parkinsons": "Parkinson"
}

In [3]:
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
    print("Old parkinson split dataset deleted.")

print("Ready to create new split dataset.")

Ready to create new split dataset.


In [4]:
random.seed(42)

for source_class, target_class in class_map.items():
    source_class_path = os.path.join(source_dir, source_class)

    files = [
        f for f in os.listdir(source_class_path)
        if f.lower().endswith(".wav")
    ]

    random.shuffle(files)

    train_end = int(0.70 * len(files))
    val_end = int(0.85 * len(files))

    splits = {
        "train": files[:train_end],
        "val": files[train_end:val_end],
        "test": files[val_end:]
    }

    for split_name, split_files in splits.items():
        split_path = os.path.join(target_dir, split_name, target_class)
        os.makedirs(split_path, exist_ok=True)

        for file in split_files:
            src = os.path.join(source_class_path, file)
            dst = os.path.join(split_path, file)
            shutil.copy(src, dst)

    print(target_class)
    print("Total:", len(files))
    print("Train:", len(splits["train"]))
    print("Val:", len(splits["val"]))
    print("Test:", len(splits["test"]))
    print()

Healthy
Total: 582
Train: 407
Val: 87
Test: 88

Parkinson
Total: 560
Train: 392
Val: 84
Test: 84



In [5]:
for split in ["train", "val", "test"]:
    print(split.upper())

    for cls in ["Healthy", "Parkinson"]:
        folder = os.path.join(target_dir, split, cls)
        count = len([f for f in os.listdir(folder) if f.lower().endswith(".wav")])
        print(cls, ":", count)

    print()

TRAIN
Healthy : 407
Parkinson : 392

VAL
Healthy : 87
Parkinson : 84

TEST
Healthy : 88
Parkinson : 84



In [6]:
train_dir = "../data/parkinson/train"
val_dir = "../data/parkinson/val"
test_dir = "../data/parkinson/test"

In [7]:
def extract_features(file_path, n_mfcc=20):
    y, sr = librosa.load(file_path, sr=16000, mono=True)

    y, _ = librosa.effects.trim(y)

    if len(y) == 0:
        raise ValueError("Empty or silent audio file.")

    y = librosa.util.normalize(y)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std  = np.std(mfcc, axis=1)

    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rms = np.mean(librosa.feature.rms(y=y))
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

    features = np.hstack([
    mfcc_mean,
    mfcc_std,
    zcr,
    rms,
    spectral_centroid
])
    return features

In [8]:
def load_dataset(data_dir):
    X = []
    y = []

    classes = sorted(os.listdir(data_dir))

    for label in classes:
        class_path = os.path.join(data_dir, label)

        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            if file.lower().endswith(".wav"):
                file_path = os.path.join(class_path, file)
                features = extract_features(file_path)

                X.append(features)
                y.append(label)

    return np.array(X), np.array(y)

In [9]:
X_train, y_train = load_dataset(train_dir)
X_val, y_val = load_dataset(val_dir)
X_test, y_test = load_dataset(test_dir)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("Classes:", np.unique(y_train))

Train shape: (799, 43)
Val shape: (171, 43)
Test shape: (172, 43)
Classes: ['Healthy' 'Parkinson']


In [10]:
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

print("Label mapping:", dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
)))

Label mapping: {np.str_('Healthy'): np.int64(0), np.str_('Parkinson'): np.int64(1)}


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [12]:
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(len(label_encoder.classes_), activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         5,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,802 (65.63 KB)

 Trainable params: 16,418 (64.13 KB)

 Non-trainable params: 384 (1.50 KB)

In [13]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
model_checkpoint = callbacks.ModelCheckpoint(
    "../backend/models/parkinson_best_model.keras",
    monitor="val_loss",
    save_best_only=True
)

early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [15]:
history = model.fit(
    X_train_scaled,
    y_train_enc,
    validation_data=(X_val_scaled, y_val_enc),
    epochs=50,
    batch_size=16,
    callbacks=[model_checkpoint, early_stopping, reduce_lr]
)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5807 - loss: 0.8428 - val_accuracy: 0.8304 - val_loss: 0.4871 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6871 - loss: 0.6149 - val_accuracy: 0.8538 - val_loss: 0.4370 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7422 - loss: 0.5638 - val_accuracy: 0.8596 - val_loss: 0.3811 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7785 - loss: 0.4825 - val_accuracy: 0.8947 - val_loss: 0.3165 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7947 - loss: 0.4633 - val_accuracy: 0.9123 - val_loss: 0.2673 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8210 - loss: 0.3909 - val_accuracy: 0.9357 - val_loss: 0.2378 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8323 - loss: 0.3813 - val_accuracy: 

In [16]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Predict on SCALED training data
y_train_pred_probs = model.predict(X_train_scaled)
y_train_pred = np.argmax(y_train_pred_probs, axis=1)

cm = confusion_matrix(y_train_enc, y_train_pred)

print("Classes:")
print(label_encoder.classes_)

print("\nConfusion Matrix (Training Set):")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_train_enc,
    y_train_pred,
    target_names=label_encoder.classes_
))

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Classes:
['Healthy' 'Parkinson']

Confusion Matrix (Training Set):
[[407   0]
 [  0 392]]

Classification Report:
              precision    recall  f1-score   support

     Healthy       1.00      1.00      1.00       407
   Parkinson       1.00      1.00      1.00       392

    accuracy                           1.00       799
   macro avg       1.00      1.00      1.00       799
weighted avg       1.00      1.00      1.00       799



In [17]:
y_val_pred_probs = model.predict(X_val_scaled)
y_val_pred = np.argmax(y_val_pred_probs, axis=1)

print("Confusion Matrix (Validation Set):")
print(confusion_matrix(y_val_enc, y_val_pred))

print("\nClassification Report:")
print(classification_report(
    y_val_enc,
    y_val_pred,
    target_names=label_encoder.classes_
))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Confusion Matrix (Validation Set):
[[86  1]
 [ 0 84]]

Classification Report:
              precision    recall  f1-score   support

     Healthy       1.00      0.99      0.99        87
   Parkinson       0.99      1.00      0.99        84

    accuracy                           0.99       171
   macro avg       0.99      0.99      0.99       171
weighted avg       0.99      0.99      0.99       171



In [18]:
best_model = keras.models.load_model("../backend/models/parkinson_best_model.keras")

In [19]:
test_loss, test_acc = best_model.evaluate(X_test_scaled, y_test_enc)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9767 - loss: 0.1732  
Test Loss: 0.17321676015853882
Test Accuracy: 0.9767441749572754


In [20]:
y_pred = best_model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_test_enc, y_pred_classes))

print("\nClassification Report:")
print(classification_report(
    y_test_enc,
    y_pred_classes,
    target_names=label_encoder.classes_
))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Confusion Matrix:
[[86  2]
 [ 2 82]]

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.98      0.98      0.98        88
   Parkinson       0.98      0.98      0.98        84

    accuracy                           0.98       172
   macro avg       0.98      0.98      0.98       172
weighted avg       0.98      0.98      0.98       172



In [21]:
best_model.save("../backend/models/parkinson_best_model.keras")
joblib.dump(scaler, "../backend/models/parkinson_scaler.pkl")
joblib.dump(label_encoder, "../backend/models/parkinson_label_encoder.pkl")

print("Saved files:")
print("../backend/models/parkinson_best_model.keras")
print("../backend/models/parkinson_scaler.pkl")
print("../backend/models/parkinson_label_encoder.pkl")

Saved files:
../backend/models/parkinson_best_model.keras
../backend/models/parkinson_scaler.pkl
../backend/models/parkinson_label_encoder.pkl


In [22]:
train_healthy_dir = "../data/parkinson/train/Healthy"

file = random.choice(os.listdir(train_healthy_dir))
file_path = os.path.join(train_healthy_dir, file)

features = extract_features(file_path)
features_scaled = scaler.transform([features])

prediction = model.predict(features_scaled)
predicted_index = np.argmax(prediction)
predicted_label = label_encoder.inverse_transform([predicted_index])[0]

print("File:", file)
print("Real label: Healthy")
print("Predicted:", predicted_label)
print("Confidence:", prediction[0][predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
File: healthy_337.wav
Real label: Healthy
Predicted: Healthy
Confidence: 0.9898571


In [23]:
import numpy as np

file_path = "../data/parkinson/test/Parkinson/parkinsons_003.wav"

print("Testing file:", file_path)

# Extract features
features = extract_features(file_path)
features_scaled = scaler.transform([features])

# --- TEST model ---
probs_model = model.predict(features_scaled, verbose=0)[0]
pred_model = label_encoder.inverse_transform([np.argmax(probs_model)])[0]

# --- TEST best_model ---
probs_best = best_model.predict(features_scaled, verbose=0)[0]
pred_best = label_encoder.inverse_transform([np.argmax(probs_best)])[0]

print("\n--- MODEL ---")
print("Prediction:", pred_model)
print("Confidence:", np.max(probs_model))

print("\n--- BEST MODEL ---")
print("Prediction:", pred_best)
print("Confidence:", np.max(probs_best))

Testing file: ../data/parkinson/test/Parkinson/parkinsons_003.wav

--- MODEL ---
Prediction: Healthy
Confidence: 0.9631208

--- BEST MODEL ---
Prediction: Healthy
Confidence: 0.9631208
